# Final PCA: per-sample-set covariate PCs

`01_ancestry_pca_filter.ipynb` decides membership (AoU's own premade PCs). This notebook only produces the covariate PCs `02_residualize_phenotypes.ipynb` reads.

Refits rather than reusing AoU's premade PCs directly -- those are fit globally across all ancestries, so PC1/PC2 there mostly separate continental groups rather than resolving structure within one final `SAMPLE_SET`. Reads `03_genome_wide_qc_thinning_batch_submit.ipynb`'s GRM panel for this `SAMPLE_SET` (already restricted to its final members -- no `--keep` needed here) and fits `--pca approx` to `N_PCS=10`.

The further `--thin` step below is now per-`SAMPLE_SET` (not shared/cached across sample sets like when there was a single ancestry panel), since each `SAMPLE_SET` now has its own separate GRM panel.

## Compute resource

8-16 vCPU is plenty -- runs on the thinned subset, not the full GRM panel.

## Setup

plink2: manual install, same pattern as everywhere else in this repo.

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink2" ]; then
  # URL is dated; if it 404s, get current link from https://www.cog-genomics.org/plink/2.0/
  PLINK2_URL="https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  cd /tmp
  wget -q -O plink2.zip "$PLINK2_URL"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink2"
fi

export PATH="$BIN_DIR:$PATH"
plink2 --version
nproc
free -h

In [ ]:
import os
import pandas as pd

bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

N_THREADS = os.cpu_count()

## Sample set configuration

`prob_tag` must match `01_ancestry_pca_filter.ipynb`'s `SAMPLE_SETS` dict -- that's what named the keep-list files.

In [ ]:
SAMPLE_SETS = {
    "eur_strict": {"prob_tag": "p10"},
    "eur_base":   {"prob_tag": "p50"},
    "eur_loose":  {"prob_tag": "p99"},
    "uniform": {"prob_tag": "uniform"},
    "afr":        {"prob_tag": "p2"},
    "eas":        {"prob_tag": "p90"},
}

N_PCS = 10              # final covariate PC count
PCA_N_SNPS_TARGET = 100_000   # GRM wants ~1M variants; PCA converges with far fewer

CDR_VERSION = "v9"

# Top-level bucket folder name for this project's outputs -- distinct from
# CDR_VERSION, which keeps its real meaning elsewhere. Fixed literal, matches
# every other notebook in this pipeline.
PROJECT_DIR = "phenotypic_covariance_v9"
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
ANCESTRY_BUCKET_DIR = f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/01_ancestry_filtering"

LOCAL_WORK_DIR = os.path.expanduser("~/scratch_grm")
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

## Inputs

Run once per `SAMPLE_SET`. Copies that `SAMPLE_SET`'s own GRM panel locally -- already restricted to its final members, no `--keep` needed.

In [ ]:
SAMPLE_SET = "eur_base"   # <-- change this and rerun for each of the 6 sample sets
_cfg = SAMPLE_SETS[SAMPLE_SET]

PANEL_DIR = f"{ANCESTRY_BUCKET_DIR}/genome_wide_panel_{SAMPLE_SET}"
MERGED_NAME = f"genome_wide_thinned_{CDR_VERSION}_{SAMPLE_SET}"   # pgen form, from 03_genome_wide_qc_thinning_batch_submit.ipynb's merge step
MERGED_PREFIX = os.path.join(LOCAL_WORK_DIR, MERGED_NAME)

for ext in ("pgen", "pvar", "psam"):
    bucket_path = f"{PANEL_DIR}/{MERGED_NAME}.{ext}"
    local_path = f"{MERGED_PREFIX}.{ext}"
    assert os.path.isfile(bucket_path), (
        f"missing GRM panel: {bucket_path!r} -- run 03_genome_wide_qc_thinning_batch_submit.ipynb's merge section first"
    )
    if not os.path.isfile(local_path) or os.path.getsize(local_path) != os.path.getsize(bucket_path):
        import shutil
        shutil.copy(bucket_path, local_path)

PCA_BED_PREFIX = os.path.join(LOCAL_WORK_DIR, f"{MERGED_NAME}_pca_thinned")   # per-SAMPLE_SET now (separate GRM panel each)

FINAL_PCA_BUCKET_DIR = f"{ANCESTRY_BUCKET_DIR}/ancestry_pca_filter/final_pca"
SAMPLE_SET_KEEP_PATH = os.path.join(FINAL_PCA_BUCKET_DIR, SAMPLE_SET, f"final_keep_ids_{SAMPLE_SET}_{_cfg['prob_tag']}.txt")
assert os.path.isfile(SAMPLE_SET_KEEP_PATH), (
    f"missing keep-list: {SAMPLE_SET_KEEP_PATH!r} -- run 01_ancestry_pca_filter.ipynb first"
)

SAMPLE_SET_OUT_DIR = os.path.join(FINAL_PCA_BUCKET_DIR, SAMPLE_SET)
os.makedirs(SAMPLE_SET_OUT_DIR, exist_ok=True)
FINAL_PCA_PREFIX = os.path.join(LOCAL_WORK_DIR, f"final_pca_{CDR_VERSION}_{SAMPLE_SET}")

print(MERGED_PREFIX)
print(SAMPLE_SET_KEEP_PATH)
print(SAMPLE_SET_OUT_DIR)

## Further prune + thin for PCA

R2-pruned first (`--indep-pairwise 50 10 0.1`, same window/step/r2 convention `explore_1kg_reference.ipynb` uses) -- PCA on unpruned data lets a few high-LD regions dominate the top PCs instead of reflecting genome-wide structure. `--thin` on top of that only if the pruned set still exceeds `PCA_N_SNPS_TARGET` (same calibrate-then-apply pattern `01_king_po_exclusion.ipynb` uses). Not shared/cached across `SAMPLE_SET`s -- each has its own separate GRM panel now, so each gets its own prune+thin.

In [ ]:
%%bash -s "$MERGED_PREFIX" "$PCA_BED_PREFIX" "$PCA_N_SNPS_TARGET" "$N_THREADS"
set -e
MERGED_PREFIX=$1
PCA_BED_PREFIX=$2
N_TARGET=$3
THREADS=$4

if [ -s "${PCA_BED_PREFIX}.pgen" ]; then
  echo "already pruned+thinned, skipping"
else
  PRUNE_PREFIX="${MERGED_PREFIX}_pruned"
  plink2 \
    --pfile "$MERGED_PREFIX" \
    --nonfounders \
    --indep-pairwise 50 10 0.1 \
    --threads "$THREADS" \
    --out "$PRUNE_PREFIX"

  N_PRUNED=$(wc -l < "${PRUNE_PREFIX}.prune.in")
  THIN_P=$(python3 -c "print(min(1.0, ${N_TARGET} / ${N_PRUNED}))")
  echo "pruned SNPs: $N_PRUNED, target: $N_TARGET, thin_p: $THIN_P"

  plink2 \
    --pfile "$MERGED_PREFIX" \
    --extract "${PRUNE_PREFIX}.prune.in" \
    --thin "$THIN_P" \
    --threads "$THREADS" \
    --make-pgen \
    --out "$PCA_BED_PREFIX"
fi

echo "Pruned+thinned SNP count:"
awk 'END{print NR-1}' "${PCA_BED_PREFIX}.pvar"

In [ ]:
%%bash -s "$PCA_BED_PREFIX" "$FINAL_PCA_PREFIX" "$N_THREADS" "$N_PCS"
set -e
PCA_BED_PREFIX=$1
FINAL_PCA_PREFIX=$2
THREADS=$3
NPCS=$4

plink2 \
  --pfile "$PCA_BED_PREFIX" \
  --nonfounders \
  --freq counts \
  --pca approx "$NPCS" allele-wts \
  --threads "$THREADS" \
  --out "$FINAL_PCA_PREFIX"

ls -lh "${FINAL_PCA_PREFIX}".*

### Scree plot

% variance explained per PC, within this `SAMPLE_SET`.

In [ ]:
import matplotlib.pyplot as plt

eigenval = pd.read_csv(f"{FINAL_PCA_PREFIX}.eigenval", header=None, names=["eigenvalue"])
eigenval["pc"] = range(1, len(eigenval) + 1)
eigenval["pct_variance"] = eigenval["eigenvalue"] / eigenval["eigenvalue"].sum() * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(eigenval["pc"], eigenval["pct_variance"], color="royalblue")
ax.set_xlabel("PC")
ax.set_ylabel("% variance explained")
ax.set_title(f"Final PCA [{SAMPLE_SET}] scree plot")
ax.set_xticks(eigenval["pc"])
plt.tight_layout()
plot_path = os.path.join(SAMPLE_SET_OUT_DIR, f"final_pca_scree_{SAMPLE_SET}.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")
print(eigenval[["pc", "eigenvalue", "pct_variance"]].to_string(index=False))

## Write PC covariates for 02_residualize_phenotypes.ipynb

`IID PC1 ... PC10` format.

In [ ]:
direct = pd.read_csv(f"{FINAL_PCA_PREFIX}.eigenvec", sep=r"\s+")

_id_col = "#IID" if "#IID" in direct.columns else "IID"
pc_cols = [c for c in direct.columns if c.startswith("PC")]
assert len(pc_cols) == N_PCS, f"expected {N_PCS} PC columns, found {len(pc_cols)}: {pc_cols}"

covariate_table = direct[[_id_col] + pc_cols].rename(columns={_id_col: "IID"})
covariate_table["IID"] = covariate_table["IID"].astype(str)

PC_COVARIATE_PATH = os.path.join(SAMPLE_SET_OUT_DIR, f"final_pca_pc_covariates_{SAMPLE_SET}.txt")
covariate_table.to_csv(PC_COVARIATE_PATH, sep="\t", index=False)
print(f"Wrote {len(covariate_table)} samples' PC1-PC{N_PCS} -> {PC_COVARIATE_PATH}")

## Write PCA loadings

`allele-wts` output (`.eigenvec.allele`) -- `ID`, `A1`, `PC1`..PC{N_PCS} allele weights. Needed to project new samples onto this exact `SAMPLE_SET`'s PC space later (e.g. out-of-sample projection), not read by `02_residualize_phenotypes.ipynb`.

In [ ]:
import shutil

LOADINGS_SRC = f"{FINAL_PCA_PREFIX}.eigenvec.allele"
LOADINGS_PATH = os.path.join(SAMPLE_SET_OUT_DIR, f"final_pca_loadings_{SAMPLE_SET}.txt")
shutil.copy(LOADINGS_SRC, LOADINGS_PATH)
print(f"Wrote loadings -> {LOADINGS_PATH}")

## Plot loadings (Manhattan-style)

Loading magnitude by genomic position, first 4 PCs -- the standard check for whether a PC is driven by a handful of variants in one region (an inversion, HLA, LCT, an under-pruned high-LD block) rather than genome-wide structure. IDs are `chrom:pos:ref:alt` (`--set-all-var-ids` in `03_genome_wide_qc_thinning_batch_submit.ipynb`), so CHROM/POS parse straight out of the ID column.

In [ ]:
N_PCS_TO_PLOT = 4

loadings = pd.read_csv(LOADINGS_PATH, sep=r"\s+")
id_col = "#ID" if "#ID" in loadings.columns else "ID"
loadings[["CHROM", "POS"]] = loadings[id_col].str.split(":", n=2, expand=True).iloc[:, :2]
loadings["CHROM"] = loadings["CHROM"].astype(str)
loadings["POS"] = loadings["POS"].astype(int)

chrom_order = [str(c) for c in range(1, 23)]
loadings = loadings[loadings["CHROM"].isin(chrom_order)].copy()
loadings["CHROM"] = pd.Categorical(loadings["CHROM"], categories=chrom_order, ordered=True)
loadings = loadings.sort_values(["CHROM", "POS"])

# cumulative x-position across chromosomes, standard Manhattan-plot construction
chrom_offsets = {}
offset = 0
chrom_centers = {}
for chrom in chrom_order:
    chrom_offsets[chrom] = offset
    chrom_max = loadings.loc[loadings["CHROM"] == chrom, "POS"].max()
    if pd.notna(chrom_max):
        chrom_centers[chrom] = offset + chrom_max / 2
        offset += chrom_max
loadings["CUM_POS"] = loadings.apply(lambda r: r["POS"] + chrom_offsets[r["CHROM"]], axis=1)

pc_cols = [f"PC{i}" for i in range(1, N_PCS_TO_PLOT + 1)]
colors = ["tab:blue", "tab:orange"]

fig, axes = plt.subplots(len(pc_cols), 1, figsize=(14, 3 * len(pc_cols)), sharex=True)
for ax, pc in zip(axes, pc_cols):
    for i, chrom in enumerate(chrom_order):
        sub = loadings[loadings["CHROM"] == chrom]
        ax.scatter(sub["CUM_POS"], sub[pc] ** 2, s=2, alpha=0.5, color=colors[i % 2])
    ax.set_ylabel(f"{pc} loading²")

axes[-1].set_xticks([chrom_centers[c] for c in chrom_order if c in chrom_centers])
axes[-1].set_xticklabels(chrom_order, fontsize=7)
axes[-1].set_xlabel("Chromosome")
fig.suptitle(f"Final PCA [{SAMPLE_SET}] loadings by genomic position")
plt.tight_layout()
plot_path = os.path.join(SAMPLE_SET_OUT_DIR, f"final_pca_loadings_manhattan_{SAMPLE_SET}.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")

## Next steps

`02_residualize_phenotypes.ipynb` already points at these output paths -- no manual copying needed.